# Chunking

**Goal:** Load the processed documentation pages from the previous notebook, split them into retrieval-ready `Chunk` objects, and inspect the results before embedding.

The `ChunkSplitter` strategy (documented in `src/chunking/chunker.py`):
1. **Primary split** on the `<!-- section/api: … -->` markers the converter embedded at every structural boundary.
2. **Merge forward** — stub segments < `min_chars` are prepended to the next segment.
3. **Sub-split** — segments > `max_chars` are sliced at paragraph boundaries with a configurable overlap tail.

## 1. Environment Setup

In [1]:
import sys
from pathlib import Path

DATA_DIR = "data"

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", sys.path[0])

from src import data_acquisition as da
from src import chunking as ck

Project root: /home/dmitry/Projects/DataScience/rag-techdoc-assistant


In [2]:
import json
import logging
from collections import Counter

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s  %(name)-20s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("notebook")

## 2. Load Saved Pages

In [3]:
OUTPUT_DIR = PROJECT_ROOT / DATA_DIR / "pytorch_docs_md"

pages = da.pipeline.load_pages_from_disk(OUTPUT_DIR)

print(f"Loaded {len(pages):,} pages from {OUTPUT_DIR}")
print(f"Example — '{pages[0].title}':  {pages[0].char_count:,} chars, {len(pages[0].symbols)} symbols")

Loaded 200 pages from /home/dmitry/Projects/DataScience/rag-techdoc-assistant/data/pytorch_docs_md
Example — 'torch.accelerator':  2,541 chars, 0 symbols


## 3. Configuration

In [4]:
splitter = ck.ChunkSplitter(
    max_chars=1500,
    min_chars=120,
    overlap_chars=200,
)

print("ChunkSplitter config:")
print(f"  max_chars     = {splitter.max_chars:,}")
print(f"  min_chars     = {splitter.min_chars:,}")
print(f"  overlap_chars = {splitter.overlap_chars:,}")

ChunkSplitter config:
  max_chars     = 1,500
  min_chars     = 120
  overlap_chars = 200


## 4. Dry Run — Single Page

In [5]:
# Pick a page with API symbols for a representative test
sample = pages[56]
sample_chunks = splitter.split(sample)

print(f"Page   : {sample.title}")
print(f"Symbols: {sample.symbols}")
print(f"Chunks : {len(sample_chunks)}")
print()

for i, chunk in enumerate(sample_chunks):
    cont = " (continuation)" if chunk.is_continuation else ""
    print(f"  [{i}] kind={chunk.kind:<12} chars={chunk.char_count:>5}  anchor={chunk.anchor}{cont}")

Page   : Automatic Mixed Precision package - torch.amp
Symbols: ['torch.amp.autocast_mode.is_autocast_available', 'torch.autocast', 'torch.amp.custom_fwd', 'torch.amp.custom_bwd', 'torch.cuda.amp.autocast', 'torch.cuda.amp.custom_fwd', 'torch.cuda.amp.custom_bwd', 'torch.cpu.amp.autocast', 'torch.cuda.amp.GradScaler', 'torch.cpu.amp.GradScaler']
Chunks : 33

  [0] kind=heading      chars= 1103  anchor=#automatic-mixed-precision-package-torch-amp
  [1] kind=heading      chars= 1179  anchor=#automatic-mixed-precision-package-torch-amp (continuation)
  [2] kind=function     chars=  450  anchor=#torch.amp.autocast_mode.is_autocast_available
  [3] kind=class        chars= 1452  anchor=#torch.autocast
  [4] kind=class        chars= 1500  anchor=#torch.autocast (continuation)
  [5] kind=class        chars=  923  anchor=#torch.autocast (continuation)
  [6] kind=class        chars= 1479  anchor=#torch.autocast (continuation)
  [7] kind=class        chars=  894  anchor=#torch.autocast (continuat

In [6]:
# Inspect a specific chunk in detail
idx = 3
c = sample_chunks[idx]
print(f"chunk_id     : {c.chunk_id}")
print(f"citation_url : {c.citation_url}")
print(f"kind         : {c.kind}")
print(f"symbol       : {c.symbol or '—'}")
print(f"params       : {c.params}")
print(f"keywords[:8] : {c.keywords[:8]}")
print()
print("─" * 60)
print(c.text)

chunk_id     : torch_autocast__0__977856
citation_url : https://docs.pytorch.org/docs/stable/amp.html#torch.autocast
kind         : class
symbol       : torch.autocast
params       : []
keywords[:8] : ['Automatic', 'Mixed', 'Precision', 'package', 'torch', 'amp', 'Autocasting', 'torch.amp.autocast_mode.is_autocast_available']

────────────────────────────────────────────────────────────
```python
classtorch.autocast(device_type, dtype=None, enabled=True, cache_enabled=None)
```
Instances of `autocast` serve as context managers or decorators that
allow regions of your script to run in mixed precision.

In these regions, ops run in an op-specific dtype chosen by autocast
to improve performance while maintaining accuracy.
See the Autocast Op Reference for details.

When entering an autocast-enabled region, Tensors may be any type.
You should not call `half()` or `bfloat16()` on your model(s) or inputs when using autocasting.

`autocast` should wrap only the forward pass(es) of your networ

## 5. Split All Pages

In [7]:
from src.chunking import split_incremental

CHUNKS_PATH = OUTPUT_DIR / "_chunks.jsonl"

all_chunks = split_incremental(
    pages=pages,
    chunks_path=CHUNKS_PATH,
    splitter=splitter,
)

print(f"\nTotal chunks : {len(all_chunks):,}")
print(f"Avg per page : {len(all_chunks) / max(len(pages), 1):.1f}")

Chunking new pages:   0%|          | 0/200 [00:00<?, ?page/s]


Total chunks : 844
Avg per page : 4.2


## 6. Quality Checks

In [8]:
import statistics

char_counts = [c.char_count for c in all_chunks]
kind_counts = Counter(c.kind for c in all_chunks)

print("Chunk size distribution:")
print(f"  min    : {min(char_counts):,} chars")
print(f"  median : {statistics.median(char_counts):,.0f} chars")
print(f"  mean   : {statistics.mean(char_counts):,.0f} chars")
print(f"  p95    : {sorted(char_counts)[int(len(char_counts)*0.95)]:,} chars")
print(f"  max    : {max(char_counts):,} chars")
print()

print("Chunk kind breakdown:")
for kind, count in kind_counts.most_common():
    bar = "█" * (count * 40 // max(kind_counts.values()))
    print(f"  {kind:<20} {count:>5}  {bar}")

oversized = [c for c in all_chunks if c.char_count > splitter.max_chars and not c.is_continuation]
if oversized:
    print(f"\n⚠  {len(oversized)} non-continuation chunks exceed max_chars (atomic code blocks).")
else:
    print(f"\n✓  All non-continuation chunks are within max_chars.")

continuation_count = sum(1 for c in all_chunks if c.is_continuation)
print(f"   Continuation sub-chunks: {continuation_count}")

Chunk size distribution:
  min    : 23 chars
  median : 504 chars
  mean   : 694 chars
  p95    : 1,692 chars
  max    : 6,328 chars

Chunk kind breakdown:
  heading                533  ████████████████████████████████████████
  object                 124  █████████
  class                   73  █████
  function                61  ████
  method                  40  ███
  attribute               13  

⚠  21 non-continuation chunks exceed max_chars (atomic code blocks).
   Continuation sub-chunks: 130


In [9]:
# Verify every chunk has a valid citation URL and non-empty text
problems = [
    c for c in all_chunks
    if not c.text.strip() or not c.citation_url
]

if problems:
    print(f"⚠  {len(problems)} chunks with missing text or citation URL:")
    for p in problems[:5]:
        print(f"   {p.chunk_id}")
else:
    print(f"✓  All {len(all_chunks):,} chunks have non-empty text and citation URLs.")

✓  All 844 chunks have non-empty text and citation URLs.


## 7. Save Chunks to Disk

Persist the chunks as a JSONL file so the embedding notebook can load them directly without re-chunking.

In [10]:
CHUNKS_PATH = OUTPUT_DIR / "_chunks.jsonl"

with CHUNKS_PATH.open("w", encoding="utf-8") as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk.to_dict(), ensure_ascii=False) + "\n")

size_mb = CHUNKS_PATH.stat().st_size / 1e6
print(f"✓ Saved {len(all_chunks):,} chunks → {CHUNKS_PATH}  ({size_mb:.1f} MB)")

✓ Saved 844 chunks → /home/dmitry/Projects/DataScience/rag-techdoc-assistant/data/pytorch_docs_md/_chunks.jsonl  (1.6 MB)
